In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parents[1]
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

EMB_IN = DATA_PROCESSED / "drhp_embeddings" / "drhp_section_embeddings.parquet"
EMB_OUT = DATA_PROCESSED / "drhp_ipo_embeddings.parquet"

print("Embedding file exists:", EMB_IN.exists())

Embedding file exists: True


In [2]:
emb = pd.read_parquet(EMB_IN)

emb.head()

,company_name,year,section,finbert_embedding,sbert_embedding
0,Rajputana Stainless Limited,2010,business,"[-0.19410531, 1.4739758, 0.06661505, -0.179569...","[0.0015715183, 0.029315984, -0.008042462, -0.0..."
1,Rajputana Stainless Limited,2010,risk,"[0.19888926, 1.2723527, -0.6584434, -0.5469876...","[-0.05275326, 0.09926484, -0.03792792, 0.02027..."
2,Rajputana Stainless Limited,2010,proceeds,"[-0.19195494, 1.582735, 0.021451645, -0.207139...","[-0.02480682, 0.021075375, -0.010292611, -0.00..."
3,The Promoters Of Our Company Are Bajaj Consume...,2010,business,"[0.41988212, 0.79313415, 0.6410486, 0.54914254...","[-0.023023352, 0.00429242, -0.07762703, -0.031..."
4,The Promoters Of Our Company Are Bajaj Consume...,2010,risk,"[0.17467354, 1.2288178, -0.68853724, -0.406259...","[-0.035153065, 0.006121801, -0.029135512, 0.02..."


In [3]:
def mean_stack(vectors):
    return np.mean(np.vstack(vectors), axis=0)

In [4]:
agg = (
    emb
    .groupby(["company_name", "section"])
    .agg(
        finbert_mean=("finbert_embedding", mean_stack),
        sbert_mean=("sbert_embedding", mean_stack)
    )
    .reset_index()
)

agg.head()

,company_name,section,finbert_mean,sbert_mean
0,Ace Tours Worldwide Limited,business,"[0.09598311, 1.5131452, -0.16303927, -0.200869...","[-0.044842836, -0.011022975, -0.0679641, -0.00..."
1,Ace Tours Worldwide Limited,proceeds,"[0.051644664, 1.4698048, -0.15920731, -0.26291...","[-0.03634191, -0.008319714, -0.068276316, -0.0..."
2,Ace Tours Worldwide Limited,risk,"[0.15280461, 1.275018, -0.62333596, -0.4211575...","[-0.03555556, 0.05038331, -0.053526152, 0.0537..."
3,Advanced Enzyme Technologies Limited,business,"[-0.015568077, 1.3338764, -0.067569315, -0.109...","[-0.043069456, -0.08231329, 0.0006061874, -0.0..."
4,Advanced Enzyme Technologies Limited,risk,"[-0.09909128, 1.2640021, -0.600338, -0.2896399...","[-0.027008343, 0.041625716, -0.040881947, 0.05..."


In [5]:
def pivot_embeddings(df, prefix):
    out = {}
    for _, row in df.iterrows():
        key = f"{prefix}_{row['section']}"
        out[key] = row[f"{prefix}_mean"]
    return out

In [6]:
ipo_rows = []

for company, grp in agg.groupby("company_name"):
    row = {"company_name": company}
    row.update(pivot_embeddings(grp, "finbert"))
    row.update(pivot_embeddings(grp, "sbert"))
    ipo_rows.append(row)

ipo_emb = pd.DataFrame(ipo_rows)
ipo_emb.shape

(272, 7)

In [7]:
ipo_emb["finbert_all"] = ipo_emb[
    [c for c in ipo_emb.columns if c.startswith("finbert_")]
].apply(lambda x: mean_stack([v for v in x if isinstance(v, np.ndarray)]), axis=1)

ipo_emb["sbert_all"] = ipo_emb[
    [c for c in ipo_emb.columns if c.startswith("sbert_")]
].apply(lambda x: mean_stack([v for v in x if isinstance(v, np.ndarray)]), axis=1)

In [8]:
ipo_emb.isna().mean().sort_values(ascending=False).head(10)

finbert_proceeds    0.257353
sbert_proceeds      0.257353
finbert_business    0.022059
sbert_business      0.022059
company_name        0.000000
finbert_risk        0.000000
sbert_risk          0.000000
finbert_all         0.000000
sbert_all           0.000000
dtype: float64